# 11. Komunikace v síti - tvorba síťových aplikací, Berkley socket a jeho rozhraní

### Princip síťové komunikace
* Zajišťuje výměnu dat mezi koncovými uzly v síti
* Například v P2P architektuře komunikují aplikace bez centrálního serveru. K tomu se využívá sada protokolů rodiny TCP/IP

### Berkley Socket
* Standardizované aplikační rozhraní (API) operačního systému, tvoří koncový bod sítě
* Umožňuje programu vysílat a přijímat data
* Socket lze použít jak pro TCP, tak i pro UDP komunikaci

### Protokoly transportní vrstvy
* TCP (Transmission Control Protocol):
    * Spojově orientovaný protokol
    * Po navázání spojení garantuje spolehlivost a doručuje data ve správném pořadí s následnou kontrolou
    * V praxi se využívá v aplikacích, kde nesmí chybět žádná data a je nutná jejich preciznost
* UDP (User Datagram Protocol):
    * Nespojovaný a velmi rychlý protokol bez záruky doručení
    * Může ztrácet pakety, po dokončení transakce neprobíhá žádná kontrola doručení
    * Pro posílání videostreamů nebo v online hrách, kde ztráta jednoho paketu nevadí

### Rozhraní a stavba síťové aplikace
* Server:
    * Vytvoří socket, sváže ho s konkrétní lokální IP adresou v síti a portem (metoda `bind()`)
    * Zahájí naslouchání na daném portu (`listen()`)
    * Přijme příchozí požadavek o spojení (`accept()`)
    * Čeká blokující metodou `recv()` na přijatá data od klienta
    * Odpovídá (`sendall()`/`send()`)
    * Po ukončení zavírá socket (`close()`)
* Klient:
    * Vytvoří socket
    * Pokusí se aktivně připojit k běžícímu serveru (`connect()`)
* Výměna dat - Probíhá pomocí metod `send()` a `recv()`. Přenáší se binární data, proto se zprávy před odesláním kódují a po přijetí dekódují na text

In [ ]:
import socket

**1. TCP SERVER**

In [ ]:
def spust_server():
    # Vytvoření síťového socketu (AF_INET = IPv4, SOCK_STREAM = TCP)
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    # Svázání s adresou (localhost) a specifickým portem (dle zadání projektu P2P)
    adresa = ('127.0.0.1', 65525)
    server_socket.bind(adresa)

    # Povolení naslouchání příchozích TCP spojení
    server_socket.listen(1)
    print(f"[SERVER] Naslouchá na portu {adresa[1]}...")

    # Blokující operace: čeká, dokud se nepřipojí klient
    klientsky_socket, adresa_klienta = server_socket.accept()
    print(f"[SERVER] Přijato spojení od: {adresa_klienta}")

    # Přijetí dat od klienta a jejich UTF-8 dekódování
    data_od_klienta = klientsky_socket.recv(1024).decode('utf-8')
    print(f"[SERVER] Přijatý příkaz: {data_od_klienta}")

    # Odeslání odpovědi (např. vrácení kódu banky v P2P síti)
    if data_od_klienta == "BC":
        odpoved = "BC 127.0.0.1"
        klientsky_socket.sendall(odpoved.encode('utf-8'))

    # Uvolnění prostředků
    klientsky_socket.close()
    server_socket.close()

**2. TCP KLIENT**

In [ ]:
def spust_klienta():
    # Vytvoření klientského socketu
    klient_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    # Aktivní připojení na IP adresu a port serveru
    print("[KLIENT] Připojuji se k serveru...")
    klient_socket.connect(('127.0.0.1', 65525))

    # Zakódování textového příkazu do bytů a odeslání
    dotaz = "BC"
    klient_socket.sendall(dotaz.encode('utf-8'))

    # Příjem odpovědi
    odpoved_byty = klient_socket.recv(1024)
    print(f"[KLIENT] Odpověď serveru: {odpoved_byty.decode('utf-8')}")

    klient_socket.close()